# Bronze: Locations Ingestion

**Purpose**: Fetch all Canadian air quality monitoring stations from OpenAQ API\
**Schedule**: Daily\
**Source**: OpenAQ API v3 `/locations` endpoint\
**Target**: `airquality.bronze.locations`

### 1.0: Configuration

In [0]:
%run ./00_utils

### 2.0: API Call Functions

In [ ]:
import time

def fetch_all_locations(countries_id):
    """
    Fetch every monitoring location for a country, page by page.

    The API returns at most 1,000 rows per page; a short page signals the
    end. Sleeps 1 second between pages to respect the rate limit (60/min).
    """
    all_locations = []
    page = 1

    while True:
        data = fetch_openaq("locations", params={
            "countries_id": countries_id,
            "limit": 1000,
            "page": page
        })

        results = data["results"]
        all_locations.extend(results)

        print(f"Page {page}: fetched {len(results)} locations (total: {len(all_locations)})")

        # A short page means this was the last one
        if len(results) < 1000:
            break

        page += 1
        time.sleep(1)

    return all_locations

### 3.0: Run ingestion

In [ ]:
# 156 is the OpenAQ country id for Canada
canada_locations = fetch_all_locations(156)
print(f"\nTotal: {len(canada_locations)} locations in Canada")

### 4.0: Clean Data

In [ ]:
import json

# Keep bronze close to the raw API payload: scalars become columns, nested
# objects are stored as JSON strings. Parsing happens in the silver notebook,
# so new fields inside these objects never break ingestion.
locations_clean = []
for loc in canada_locations:
    clean = {
        "id": loc["id"],
        "name": loc.get("name"),
        "locality": loc.get("locality"),
        "timezone": loc.get("timezone"),
        "isMobile": loc.get("isMobile"),
        "isMonitor": loc.get("isMonitor"),
        # Nested objects as JSON strings
        "country": json.dumps(loc.get("country")),
        "owner": json.dumps(loc.get("owner")),
        "provider": json.dumps(loc.get("provider")),
        "instruments": json.dumps(loc.get("instruments")),
        "sensors": json.dumps(loc.get("sensors")),
        "coordinates": json.dumps(loc.get("coordinates")),
        "licenses": json.dumps(loc.get("licenses")),
        "bounds": json.dumps(loc.get("bounds")),
        "datetimeFirst": json.dumps(loc.get("datetimeFirst")),
        "datetimeLast": json.dumps(loc.get("datetimeLast")),
    }
    locations_clean.append(clean)

print(f"Prepared {len(locations_clean)} locations for bronze")

### 5.0: Save Locations to Bronze

In [ ]:
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime, timezone

# Convert to Spark and stamp every row with load metadata:
# ingested_at lets silver pick the latest snapshot of each location.
df = spark.createDataFrame(pd.DataFrame(locations_clean))

df_bronze = df.withColumn("ingested_at", F.lit(datetime.now(timezone.utc).isoformat())) \
              .withColumn("source", F.lit("openaq_api_v3"))

# Append only: each run adds a full snapshot; bronze is the immutable
# history the pipeline can always rebuild from.
df_bronze.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.locations")

print(f"Appended {len(locations_clean)} locations to {CATALOG}.{SCHEMA}.locations")